In [2]:
from google.colab import files
uploaded = files.upload()

Saving FINC3017_A2_Workbook (2).xlsx to FINC3017_A2_Workbook (2).xlsx


In [3]:
import io
import openpyxl

filename = next(iter(uploaded))

wb = openpyxl.load_workbook(
    io.BytesIO(uploaded[filename]),
    data_only=True
)

print("Workbook loaded:", filename)
print("Sheets:", wb.sheetnames)

Workbook loaded: FINC3017_A2_Workbook (2).xlsx
Sheets: ['README', 'Q3b(ii)', 'Data_FF5', 'Data_SBM25', 'Data_OPINV25', 'Data_IND17', 'Data_IND49', 'Data_MktCap', 'XR_SBM25', 'XR_OPINV25', 'XR_IND17', 'XR_IND49', 'FM_Betas_SBM25', 'FM_CS_SBM25', 'FM_Betas_OPINV25', 'FM_CS_OPINV25', 'FM_Betas_IND17', 'FM_CS_IND17', 'T3_Inputs', 'T3_Sigma', 'T3_Portfolios', 'T3_BlackLitterman', 'T3_BL_Sensitivity', 'T4_Backtest', 'R_Robustness', 'Verification', 'Charts']


In [4]:
# Check calculation inputs
import numpy as np
from scipy.optimize import minimize

def read_range(sheet, r1, r2, c1, c2):
    values = [
        [wb[sheet].cell(r, c).value
         for c in range(c1, c2 + 1)]
        for r in range(r1, r2 + 1)
    ]
    if any(v is None for row in values for v in row):
        raise ValueError(
            f"Missing saved values in {sheet}. "
            "Recalculate and save in Excel, then upload again."
        )
    result = np.array(values, dtype=float)
    if not np.all(np.isfinite(result)):
        raise ValueError(f"Invalid numbers in {sheet}.")
    return result

sigma = read_range("T3_Sigma", 4, 52, 2, 50)
prior = read_range("T3_BlackLitterman", 9, 57, 4, 4).ravel()
betas = read_range("T3_Inputs", 11, 59, 3, 7)[:, ::-1]
premiums = read_range("T3_Inputs", 4, 4, 3, 7).ravel()

names = [
    wb["T3_Inputs"].cell(r, 1).value
    for r in range(11, 60)
]

print("Inputs loaded successfully.")
print("Industries:", len(names))
print("Market premium:", premiums[0])
print("RMW premium:", premiums[3])

Inputs loaded successfully.
Industries: 49
Market premium: 0.004669679192183511
RMW premium: 0.004762573274751897


In [6]:
# Rows: Market view, RMW view.
# Columns: the 49 industries.
P = np.zeros((2, 49))

for row, factor in enumerate([0, 3]):
    ranked = np.argsort(betas[:, factor])

    P[row, ranked[-10:]] = 0.1   # Highest ten loadings
    P[row, ranked[:10]] = -0.1   # Lowest ten loadings

Q = premiums[[0, 3]]

print("Market loading difference:", P[0] @ betas[:, 0])
print("RMW loading difference:", P[1] @ betas[:, 3])

Market loading difference: 0.6540515039536383
RMW loading difference: 1.030332448343605


In [7]:
# calculate the baseline BL expected returns - T3_BlackLitterman!I9
tau = 1 / 60

def posterior_returns(confidence):
    prior_uncertainty = tau * sigma
    view_covariance = P @ prior_uncertainty @ P.T

    omega = np.diag(np.diag(view_covariance)) / confidence

    return (
        prior
        + prior_uncertainty @ P.T
        @ np.linalg.solve(
            view_covariance + omega,
            Q - P @ prior
        )
    )

posterior = posterior_returns(confidence=1)

print("First industry's posterior return:", posterior[0])

First industry's posterior return: 0.005064262417901321


In [8]:
#Calculate baseline portfolio weights - T3_Portfolios!G5:G53
def optimise(mu):
    def objective(w):
        return -(w @ mu) / np.sqrt(w @ sigma @ w)

    def gradient(w):
        variance = w @ sigma @ w
        return (
            -mu / np.sqrt(variance)
            + (w @ mu) * (sigma @ w) / variance**1.5
        )

    result = minimize(
        objective,
        np.ones(49) / 49,
        jac=gradient,
        method="SLSQP",
        bounds=[(0.001, 1)] * 49,
        constraints={
            "type": "eq",
            "fun": lambda w: w.sum() - 1,
            "jac": lambda w: np.ones(49)
        },
        options={"ftol": 1e-14, "maxiter": 2000}
    )

    if not result.success:
        raise RuntimeError(result.message)

    return result.x

baseline_weights = optimise(posterior)

print("Gold weight:", baseline_weights[names.index("Gold")])
print(
    "Monthly posterior Sharpe:",
    (baseline_weights @ posterior)
    / np.sqrt(baseline_weights @ sigma @ baseline_weights)
)
print("Sum of weights:", baseline_weights.sum())
print("Minimum weight:", baseline_weights.min())

Gold weight: 0.11988361043634171
Monthly posterior Sharpe: 0.1588173635984097
Sum of weights: 1.0000000000000002
Minimum weight: 0.001


In [9]:
# reproduce the confidence sensitivity - T3_BL_Sensitivity!A1:C4
sensitivity_weights = {}

print("Confidence multiplier\tRelative view-error variance\tGold weight")

for c in [0.25, 1, 4]:
    scenario_returns = posterior_returns(confidence=c)
    weights = optimise(scenario_returns)
    sensitivity_weights[c] = weights

    gold_weight = weights[names.index("Gold")]

    print(f"{c}\t{1/c:g}\t{gold_weight:.6%}")

Confidence multiplier	Relative view-error variance	Gold weight
0.25	4	4.555356%
1	1	11.988361%
4	0.25	18.779393%


In [10]:
# CC check - Verification!B21:D21
import cvxpy as cp

# Use the baseline posterior returns and covariance matrix.
mu = posterior_returns(confidence=1)

# Charnes-Cooper-style transformation:
# y = weights / expected portfolio excess return.
# Minimise variance while fixing mu @ y = 1.
y = cp.Variable(49)

problem = cp.Problem(
    cp.Minimize(cp.quad_form(y, cp.psd_wrap(sigma))),
    [
        mu @ y == 1,
        y >= 0.001 * cp.sum(y)
    ]
)

problem.solve(
    solver="CLARABEL",
    tol_gap_abs=1e-10,
    tol_gap_rel=1e-10,
    tol_feas=1e-10,
    max_iter=1000
)

if problem.status != "optimal":
    raise RuntimeError(
        "Check not completed. Solver status: " + problem.status
    )

cc_weights = y.value / y.value.sum()

if (
    not np.all(np.isfinite(cc_weights))
    or abs(cc_weights.sum() - 1) > 1e-8
    or cc_weights.min() < 0.001 - 1e-8
):
    raise RuntimeError("CC weights failed the constraint checks.")

cc_sharpe = (
    cc_weights @ mu
) / np.sqrt(cc_weights @ sigma @ cc_weights)

slsqp_sharpe = (
    baseline_weights @ mu
) / np.sqrt(baseline_weights @ sigma @ baseline_weights)

print("CC monthly Sharpe:", f"{cc_sharpe:.12f}")
print("SLSQP monthly Sharpe:", f"{slsqp_sharpe:.12f}")
print("Absolute difference:", abs(slsqp_sharpe - cc_sharpe))


CC monthly Sharpe: 0.158817363598
SLSQP monthly Sharpe: 0.158817363598
Absolute difference: 1.593447596093256e-13


In [11]:
# Output goes in Excel:
# 2025 monthly total returns - BL_Sensitivity!D1:D4

import numpy as np

ws = wb["Data_IND49"]

# Match return columns to the industry order used in optimisation
headers = [ws.cell(3, col).value for col in range(3, 52)]
columns = [headers.index(name) + 3 for name in names]

rows = [
    row for row in range(4, ws.max_row + 1)
    if str(ws.cell(row, 1).value).startswith("2025-")
]
assert len(rows) == 12, "Expected 12 months of 2025 returns."

returns_2025 = np.array([
    [ws.cell(row, col).value for col in columns]
    for row in rows
], dtype=float)

print("2025 portfolio return")
for c in [0.25, 1, 4]:
    weights = np.asarray(sensitivity_weights[c]).reshape(-1)
    monthly_returns = returns_2025 @ weights
    annual_return = np.prod(1 + monthly_returns) - 1
    print(f"Confidence {c:g}: {annual_return:.4%}")

2025 portfolio return
Confidence 0.25: 19.1079%
Confidence 1: 24.6444%
Confidence 4: 32.1673%


In [12]:
# optimise using Σ + M
import numpy as np
from scipy.optimize import minimize

# Posterior mean uncertainty at baseline confidence c = 1
T = tau * sigma
V = P @ T @ P.T
Omega = np.diag(np.diag(V))

M = T - T @ P.T @ np.linalg.solve(V + Omega, P @ T)
predictive_sigma = sigma + M
predictive_sigma = (predictive_sigma + predictive_sigma.T) / 2

mu = np.asarray(posterior).reshape(-1)
n = len(mu)

def objective(w):
    return -(w @ mu) / np.sqrt(w @ predictive_sigma @ w)

def gradient(w):
    variance = w @ predictive_sigma @ w
    return (
        -mu / np.sqrt(variance)
        + (w @ mu) * (predictive_sigma @ w) / variance**1.5
    )

result = minimize(
    objective,
    np.asarray(baseline_weights).reshape(-1),
    jac=gradient,
    method="SLSQP",
    bounds=[(0.001, 1)] * n,
    constraints=[{
        "type": "eq",
        "fun": lambda w: w.sum() - 1,
        "jac": lambda w: np.ones(n)
    }],
    options={"ftol": 1e-14, "maxiter": 2000}
)

assert result.success, result.message
assert abs(result.x.sum() - 1) < 1e-8
assert result.x.min() >= 0.001 - 1e-8

gold_index = list(names).index("Gold")

print(f"Gold using Sigma:     {baseline_weights[gold_index]:.4%}")
print(f"Gold using Sigma + M: {result.x[gold_index]:.4%}")
print(
    "Change:",
    f"{100 * (result.x[gold_index] - baseline_weights[gold_index]):.4f}",
    "percentage points"
)

Gold using Sigma:     11.9884%
Gold using Sigma + M: 11.8801%
Change: -0.1083 percentage points
